# 03 — LUNA16 classical ML (SVM / RF / KNN)

5-fold stratified outer CV with inner 3-fold grid search per model. Both default-threshold and Youden-calibrated metrics are saved per fold to `results/luna16_<model>.json`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import yaml

from utils.seed import set_seed
from utils.models import CLASSICAL_REGISTRY
from utils.training import train_classical_cv, save_fold_results
from utils.metrics import aggregate_folds
from utils.stats import format_results_table

with open('../configs/luna16.yaml') as f:
    cfg = yaml.safe_load(f)
set_seed(cfg['seed'])

cache_dir = Path('..') / cfg['data']['cache_dir']
data = np.load(cache_dir / 'features.npz', allow_pickle=True)
X, y = data['X'], data['y']
groups = data['seriesuid']  # GroupKFold by CT scan to avoid patch-level leakage
print('X', X.shape, 'pos rate:', y.mean())

X (448, 91) pos rate: 0.25


In [2]:
results_dir = Path('..') / cfg['paths']['results']
results_dir.mkdir(parents=True, exist_ok=True)

aggregated = {}
for name, (builder, grid) in CLASSICAL_REGISTRY.items():
    print(f'\n=== {name} ===')
    folds = train_classical_cv(
        X, y, builder, grid,
        n_splits=cfg['cv']['n_splits'],
        inner_splits=cfg['cv']['inner_splits'],
        scoring=cfg['cv']['scoring'],
        seed=cfg['seed'],
        groups=groups,
        verbose=True,
    )
    save_fold_results(folds, results_dir / f'luna16_{name}.json')
    aggregated[name] = aggregate_folds([f.metrics_calibrated for f in folds])


=== SVM ===
[fold 0] train=356 test=92


[fold 0] best_params={'C': 10, 'gamma': 'scale', 'kernel': 'rbf'} thr=0.289 AUC=0.749 BalAcc(cal)=0.667
[fold 1] train=356 test=92
[fold 1] best_params={'C': 1, 'gamma': 0.01, 'kernel': 'rbf'} thr=0.202 AUC=0.758 BalAcc(cal)=0.732
[fold 2] train=360 test=88
[fold 2] best_params={'C': 10, 'gamma': 0.1, 'kernel': 'rbf'} thr=0.362 AUC=0.806 BalAcc(cal)=0.689
[fold 3] train=360 test=88
[fold 3] best_params={'C': 1, 'gamma': 0.1, 'kernel': 'rbf'} thr=0.354 AUC=0.758 BalAcc(cal)=0.659
[fold 4] train=360 test=88
[fold 4] best_params={'C': 10, 'gamma': 'scale', 'kernel': 'rbf'} thr=0.312 AUC=0.736 BalAcc(cal)=0.621

=== RF ===
[fold 0] train=356 test=92
[fold 0] best_params={'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 500} thr=0.619 AUC=0.761 BalAcc(cal)=0.536
[fold 1] train=356 test=92
[fold 1] best_params={'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100} thr=0.506 AUC=0.763 BalAcc(cal)=0.688
[fold 2] train=360 test=88
[fold 2] best_params={'max_depth': 10, 'min_samp

In [3]:
table = format_results_table(aggregated)
table

,accuracy,sensitivity,specificity,precision,f1,balanced_accuracy,auc_roc,pr_auc
model,,,,,,,,
SVM,0.717 ± 0.062,0.587 ± 0.187,0.761 ± 0.136,0.478 ± 0.111,0.506 ± 0.047,0.674 ± 0.041,0.761 ± 0.027,0.514 ± 0.066
RF,0.745 ± 0.023,0.321 ± 0.220,0.887 ± 0.094,0.550 ± 0.112,0.346 ± 0.176,0.604 ± 0.066,0.755 ± 0.016,0.516 ± 0.033
KNN,0.653 ± 0.069,0.585 ± 0.351,0.676 ± 0.206,0.303 ± 0.171,0.395 ± 0.222,0.630 ± 0.075,0.743 ± 0.022,0.461 ± 0.043


In [4]:
import json

for name, (_, grid) in CLASSICAL_REGISTRY.items():
    folds_path = results_dir / f'luna16_{name}.json'
    with folds_path.open() as f:
        raw = json.load(f)
    print(name, 'selected hparams per fold:')
    for r in raw:
        print(' ', r['fold'], r['best_params'])

SVM selected hparams per fold:
  0 {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
  1 {'C': 1, 'gamma': 0.01, 'kernel': 'rbf'}
  2 {'C': 10, 'gamma': 0.1, 'kernel': 'rbf'}
  3 {'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}
  4 {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
RF selected hparams per fold:
  0 {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 500}
  1 {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}
  2 {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 200}
  3 {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 500}
  4 {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 500}
KNN selected hparams per fold:
  0 {'n_neighbors': 21, 'p': 2, 'weights': 'uniform'}
  1 {'n_neighbors': 21, 'p': 2, 'weights': 'uniform'}
  2 {'n_neighbors': 21, 'p': 1, 'weights': 'distance'}
  3 {'n_neighbors': 21, 'p': 1, 'weights': 'uniform'}
  4 {'n_neighbors': 9, 'p': 1, 'weights': 'uniform'}
